# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 04: Advanced Chunking Strategies & Contextual Retrieval

### 1. Conceptual Intuition & Real-World Use Case
Naive chunking (e.g. fixed 500 characters with 50 character overlap) frequently ruins RAG pipelines:
- It chops sentences in half, severing semantic meaning.
- Chunks taken from the middle of a 50-page PDF lose document-level context (e.g., a table row listing `Revenue: +15%` is useless if the reader doesn't know it refers to "Q3 2024 Cloud Division").

Two state-of-the-art solutions:
1. **Semantic Chunking:** Computes embeddings of sequential sentences and places chunk boundaries at points of semantic divergence (drops in cosine similarity).
2. **Parent-Child (Hierarchical) Indexing:** Small child chunks are indexed in the vector store for precision embedding matching, but the larger parent context is fed into the LLM prompt.

---

### 2. Mathematical Foundations: Semantic Splitting
Given a sequence of sentences $S = (s_1, s_2, \dots, s_n)$ with embeddings $e_i$:

1. Compute adjacent cosine distance:
   $$d_i = 1 - \frac{e_i \cdot e_{i+1}}{\|e_i\| \|e_{i+1}\|}$$
2. Calculate moving threshold (e.g. 85th percentile of distances across the document):
   $$\tau = \mu_d + k \cdot \sigma_d$$
3. Split wherever $d_i > \tau$.

In [ ]:
import numpy as np

# Step 1: Document containing distinct thematic sections
sentences = [
    # Topic 1: Machine Learning Models
    "Linear regression models the relationship between dependent and independent features.",
    "Logistic regression extends this framework to binary classification problems.",
    "Decision trees split feature spaces recursively based on information gain or Gini impurity.",
    # Topic 2: Database Storage Engines (Sudden topic shift!)
    "PostgreSQL utilizes Multi-Version Concurrency Control to handle parallel transactions.",
    "Write-Ahead Logging guarantees data integrity in the event of hardware server crashes.",
    "B-Tree indexes provide logarithmic time lookups for ordered primary key scans."
]

# Simulate 8-dimensional sentence embeddings
np.random.seed(42)
# Create 2 distinct semantic centroids
centroid_ml = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)
centroid_db = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

embeddings = []
for i, s in enumerate(sentences):
    base = centroid_ml if i < 3 else centroid_db
    noisy = base + np.random.randn(8) * 0.15
    embeddings.append(noisy / np.linalg.norm(noisy))
embeddings = np.array(embeddings)

# Step 2: Compute Adjacent Cosine Distances
distances = []
for i in range(len(embeddings) - 1):
    cos_sim = np.dot(embeddings[i], embeddings[i + 1])
    distances.append(1.0 - cos_sim)

print("Adjacent Sentence Cosine Distances:")
for i, dist in enumerate(distances):
    print(f"  Sentence {i} -> {i+1}: Distance = {dist:.4f}")

# Step 3: Detect Semantic Breakpoint
split_threshold = np.mean(distances) + 0.5 * np.std(distances)
print(f"\nAdaptive Threshold (mu + 0.5*sigma): {split_threshold:.4f}")

chunks = []
current_chunk = [sentences[0]]

for i, dist in enumerate(distances):
    if dist > split_threshold:
        # Split point detected!
        chunks.append(current_chunk)
        current_chunk = [sentences[i + 1]]
    else:
        current_chunk.append(sentences[i + 1])
chunks.append(current_chunk)

print(f"\nGenerated {len(chunks)} Semantic Chunks:")
for idx, chunk in enumerate(chunks, 1):
    print(f"\nChunk {idx} ({len(chunk)} sentences):")
    for s in chunk:
        print(f"  - {s}")

### 3. Step-by-Step Code Walkthrough
1. **Sentence Embedding Distances:** Distances between related sentences (0 $\to$ 1, 1 $\to$ 2) are low ($\approx 0.02$).
2. **Semantic Divergence Detection:** At sentence 2 $\to$ 3 (shift from ML to Databases), distance spikes to $> 0.80$.
3. **Adaptive Splitting:** The algorithm automatically creates two coherent, non-fragmented chunks without hardcoding fixed character lengths.

---

### 4. Industry Applications & Trade-offs
- **Anthropic Contextual Retrieval:** Prepends an AI-generated 50-word context summary (e.g. "From ACME Corp Q3 Financial Report, Section 4.2") to each chunk before embedding, drastically boosting retrieval recall.
- **Trade-off:** Semantic chunking requires generating embeddings for every sentence upfront, increasing preprocessing compute.

---

### 5. Top Interview Questions & Answers
**Q1: What is the Parent-Child document retrieval pattern?**
> **Answer:** Embedding large 2,000-token chunks dilutes specific facts and degrades retrieval accuracy. Conversely, passing tiny 100-token chunks to an LLM lacks sufficient context to answer questions. Parent-Child indexing embeds small child chunks (e.g. 150 tokens) for search, but retrieves the larger parent section (e.g. 1,000 tokens) to supply to the LLM.

**Q2: What is the impact of chunk overlap in sliding window chunking?**
> **Answer:** Chunk overlap (typically 10% - 20%) ensures concepts and named entities spanning chunk boundaries are not truncated. However, excessive overlap inflates vector storage costs and leads to duplicated retrieved context.